# 03 · Treino e competição de modelos

**Feature F5** (critério 5) · dono: ver `config/equipe.yaml` · **status: F5 pronta; o registro no MLflow depende da F6**

| | |
|---|---|
| **Entradas** | `ciclo_atual.json`, `features_eleitas.json` (notebook 02), `dev.parquet`, `cfg['modelos']` |
| **Saídas** | `models/candidatos/<cycle_id>/modelo_{A,B}.joblib` · `data/processed/resultados_competicao.json` · `evidencias/competicao/resultados_<cycle_id>.json` · 2 runs no MLflow |

**Regras do enunciado que se aplicam**
- Dois algoritmos distintos sobre as **mesmas** features eleitas, mesma partição e mesma métrica.
- Folds de `dividir_cv` e métricas de `metricas_classificacao`, os mesmos de todo o pipeline.
- Registrar hiperparâmetros, seeds, métricas, features e artefatos no MLflow com o mesmo `cycle_id`.
- O teste fica de fora: só aparece no notebook 04.

Contrato completo na docstring de `src/modelagem.py`.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))

from src.config import carregar_config, carregar_contexto, resolver
from src.preprocessamento import carregar_particao
from src.modelagem import (
    avaliar_cv, construir_pipeline, eleger_por_fold, montar_resultados,
    salvar_modelos, salvar_resultados, treinar_e_avaliar,
)

cfg = carregar_config()
ctx = carregar_contexto(cfg, RAIZ)  # escrito pelo notebook 01
print(f"cycle_id={ctx['cycle_id']} | data_version={ctx['data_version']}")

cycle_id=20261119T175604Z-local | data_version=dv018-f95c3940


## Entradas
As features eleitas precisam ser do mesmo ciclo. Se o notebook 02 não rodou depois do 01, o pipeline para aqui em vez de treinar com um conjunto de outro ciclo.

In [2]:
eleicao = json.loads((resolver(cfg["dados"]["dir_processed"], RAIZ) / "features_eleitas.json").read_text(encoding="utf-8"))
for chave in ("cycle_id", "data_version"):
    if eleicao[chave] != ctx[chave]:
        raise ValueError(f"features_eleitas.json é de outro ciclo ({chave}: {eleicao[chave]} != {ctx[chave]}). Rode o notebook 02.")
eleitas = eleicao["features_eleitas"]

dev = carregar_particao("dev", cfg, RAIZ)
metrica = cfg["avaliacao"]["metrica_principal"]
print(f"dev: {len(dev):,} linhas | {cfg['avaliacao']['cv']['n_splits']} folds cronológicos | métrica: {metrica}")
print(f"{len(eleitas)} features eleitas: {eleitas}")
for nome, spec in cfg["modelos"].items():
    print(f"Modelo {nome}: {spec['algoritmo']} {spec.get('params', {})}")

dev: 62,807 linhas | 4 folds cronológicos | métrica: f1_macro
17 features eleitas: ['razao_cancel_previo', 'deposit_type_No Deposit', 'deposit_type_Non Refund', 'required_car_parking_spaces', 'previous_cancellations', 'market_segment_Groups', 'lead_time', 'total_of_special_requests', 'hotel_City Hotel', 'agent_1', 'distribution_channel_Direct', 'arrival_date_week_number', 'valor_estimado_estadia', 'total_noites', 'adr', 'customer_type_Transient', 'stays_in_week_nights']
Modelo A: logistic_regression {'C': 1.0, 'max_iter': 1000, 'class_weight': 'balanced'}
Modelo B: random_forest {'n_estimators': 300, 'max_depth': 16, 'min_samples_leaf': 5, 'class_weight': 'balanced_subsample', 'n_jobs': -1}


## Competição
Cada modelo é um único `Pipeline`: pré-processador, seleção das eleitas e algoritmo. Em cada fold o pipeline inteiro é reajustado só com o treino daquele fold, então winsorização, escala e one-hot nunca veem a validação. Depois do CV, cada um é treinado no dev inteiro, e é essa versão que vai para `models/candidatos/`.

Os hiperparâmetros são os de `config.yaml`, fixados antes do primeiro treino. Não houve ajuste fino olhando o resultado.

In [3]:
pipelines = {nome: construir_pipeline(nome, eleitas, cfg) for nome in cfg["modelos"]}
avaliacoes = {nome: treinar_e_avaliar(pipe, dev, cfg) for nome, pipe in pipelines.items()}

resumo = pd.DataFrame({
    nome: {m: f"{a['media'][m]:.4f} ± {a['desvio'][m]:.4f}" for m in a["media"]}
    for nome, a in avaliacoes.items()
})
resumo.columns = [f"{n} · {cfg['modelos'][n]['algoritmo']}" for n in resumo.columns]
display(resumo.rename_axis("média ± desvio entre folds"))

,A · logistic_regression,B · random_forest
média ± desvio entre folds,,
f1_macro,0.7406 ± 0.0133,0.7786 ± 0.0497
balanced_accuracy,0.7398 ± 0.0241,0.7624 ± 0.0453
recall_cancelamento,0.6089 ± 0.1372,0.5956 ± 0.0830
precision_cancelamento,0.7385 ± 0.1490,0.8215 ± 0.0644
roc_auc,0.8275 ± 0.0164,0.8483 ± 0.0258
pr_auc,0.7897 ± 0.0361,0.8116 ± 0.0414


In [4]:
por_fold = pd.concat(
    [pd.DataFrame(a["por_fold"]).assign(modelo=nome) for nome, a in avaliacoes.items()]
).pivot(index="fold", columns="modelo", values=metrica)
por_fold["diferenca_B_menos_A"] = por_fold["B"] - por_fold["A"]
display(por_fold.round(4).rename_axis(f"{metrica} por fold"))
print("B vence em", int((por_fold["diferenca_B_menos_A"] > 0).sum()), "de", len(por_fold), "folds")

modelo,A,B,diferenca_B_menos_A
f1_macro por fold,,,
1,0.7380,0.8438,0.1057
2,0.7268,0.7423,0.0155
3,0.7587,0.7909,0.0322
4,0.7389,0.7373,-0.0016


B vence em 3 de 4 folds


### Features eleitas ausentes em algum fold
Quando uma categoria fica abaixo de 1% no treino de um fold, o one-hot não cria a coluna. A coluna entra zerada (é o que o one-hot faria com a categoria ausente), e os dois modelos continuam vendo as mesmas colunas.

In [5]:
for nome, a in avaliacoes.items():
    print(f"Modelo {nome}:", a["features_ausentes"] or "nenhuma ausente nos folds")

Modelo A: nenhuma ausente nos folds
Modelo B: nenhuma ausente nos folds


## Quanto o CV está otimista
As eleitas vieram da média dos rankings dos 4 folds. Como o treino dos folds finais contém a validação dos iniciais, o conjunto eleito já "viu" parte das linhas em que é avaliado.

Para medir isso, refaço a votação da F4 com o ranking de **cada fold isolado** (arquivo `rankings_por_fold` da F3) e avalio cada fold com o conjunto eleito só pelo seu próprio treino. Se o CV oficial sair acima do CV com eleição por fold, a diferença estima o otimismo. Se sair abaixo, o reaproveitamento das validações não está inflando o resultado; o conjunto de cada fold só se adapta melhor ao próprio período. O CV oficial continua sendo o que vai para a F8.

In [6]:
caminho_por_fold = resolver("evidencias/selecao", RAIZ) / f"rankings_por_fold_{ctx['cycle_id']}.csv"
rankings_fold = pd.read_csv(caminho_por_fold)
eleitas_fold = eleger_por_fold(rankings_fold, cfg)
for fold, lista in eleitas_fold.items():
    print(f"fold {fold}: {len(lista)} eleitas | fora do conjunto oficial: {sorted(set(lista) - set(eleitas))} | faltando: {sorted(set(eleitas) - set(lista))}")

sensibilidade = {
    nome: avaliar_cv(construir_pipeline(nome, eleitas, cfg), dev, cfg, features_por_fold=eleitas_fold)
    for nome in cfg["modelos"]
}
comparacao = pd.DataFrame({
    nome: {
        "CV oficial (conjunto do ciclo)": avaliacoes[nome]["media"][metrica],
        "CV com eleição por fold": sensibilidade[nome]["media"][metrica],
    }
    for nome in cfg["modelos"]
}).T
comparacao["diferenca (oficial - por fold)"] = comparacao.iloc[:, 0] - comparacao.iloc[:, 1]
display(comparacao.round(4).rename_axis(metrica))

fold 1: 17 eleitas | fora do conjunto oficial: ['adults', 'market_segment_Direct', 'total_hospedes'] | faltando: ['arrival_date_week_number', 'customer_type_Transient', 'hotel_City Hotel']
fold 2: 16 eleitas | fora do conjunto oficial: ['hotel_Resort Hotel'] | faltando: ['customer_type_Transient', 'distribution_channel_Direct']
fold 3: 16 eleitas | fora do conjunto oficial: ['agent_9'] | faltando: ['distribution_channel_Direct', 'stays_in_week_nights']
fold 4: 18 eleitas | fora do conjunto oficial: ['agent_9', 'distribution_channel_TA/TO'] | faltando: ['distribution_channel_Direct']


,CV oficial (conjunto do ciclo),CV com eleição por fold,diferenca (oficial - por fold)
f1_macro,,,
A,0.7406,0.7759,-0.0353
B,0.7786,0.7848,-0.0063


## Saídas
Os dois pipelines ajustados no dev inteiro e o json com as métricas, que o notebook 04 (F8) lê para eleger o vencedor e decidir a promoção.

In [7]:
caminhos_modelos = salvar_modelos(pipelines, ctx, RAIZ)
resultados = montar_resultados(ctx, cfg, eleitas, avaliacoes, sensibilidade, caminhos_modelos={n: p.relative_to(RAIZ) for n, p in caminhos_modelos.items()})
caminhos_resultados = salvar_resultados(resultados, cfg, RAIZ)
for caminho in [*caminhos_modelos.values(), *caminhos_resultados.values()]:
    print(caminho.relative_to(RAIZ))

models/candidatos/20261119T175604Z-local/modelo_A.joblib
models/candidatos/20261119T175604Z-local/modelo_B.joblib
data/processed/resultados_competicao.json
evidencias/competicao/resultados_20261119T175604Z-local.json


## Registro no MLflow (F6)
Um run por modelo, com as mesmas tags de ciclo. A conexão com o MLflow é da F6 (`src/rastreio.py`); enquanto ela não estiver pronta, esta célula avisa e segue, e tudo o que seria registrado já está no json acima.

In [8]:
import mlflow

from src import rastreio

run_ids = {}
try:
    for nome, item in resultados["modelos"].items():
        with rastreio.iniciar_run(f"03_modelo_{nome}", ctx, cfg) as run:
            mlflow.log_params({"modelo": nome, "algoritmo": item["algoritmo"], "seed": item["seed"],
                               "n_features": len(eleitas), **item["params"]})
            mlflow.log_metrics({f"cv_{m}": v for m, v in item["cv"]["media"].items()})
            mlflow.log_dict({"features_eleitas": eleitas}, "features_eleitas.json")
            rastreio.registrar_artefatos([caminhos_modelos[nome], caminhos_resultados["resultados"]], cfg)
            run_ids[nome] = run.info.run_id
    print("runs:", run_ids)
except NotImplementedError:
    print("AVISO: src/rastreio.py (F6) ainda não foi implementado; os runs não foram registrados.")

AVISO: src/rastreio.py (F6) ainda não foi implementado; os runs não foram registrados.


## Limitações
- **Otimismo do CV:** medido acima, comparando com a eleição feita fold a fold. De qualquer forma, o número que vale para a decisão final é o do teste, no notebook 04.
- **Limiar fixo em 0,5:** a `class_weight` balanceada já compensa o desbalanceamento, mas o limiar não foi ajustado. Ajustá-lo olhando a validação seria mais uma escolha feita depois de ver resultados.
- **Hiperparâmetros sem busca:** os dois modelos rodam com a proposta inicial de `config.yaml`. A competição compara os dois algoritmos nessas configurações, não o melhor possível de cada um.
- **Desvio entre folds:** com 4 folds, diferenças de média menores que o desvio não indicam um vencedor claro. A regra de promoção da F8 (`min_melhoria_abs`) existe para isso.